# 02 — Tire Degradation Model
Goal: predict lap-time delta caused by tire age, given compound and circuit.

This notebook exists to work out the **leakage story by hand, once** — see it
through, understand why the naive number is wrong, then `ml/tire_degradation/train.py`
is what actually gets scheduled. Don't skip to `train.py` without running this first.


In [11]:
from pathlib import Path
import sys
import os

def find_backend_dir(start: Path) -> Path:
    """Walk upward from `start` until we find a directory named 'backend'
    that actually contains ml/, ingestion/, and .env.example — this makes
    the notebook runnable regardless of the cwd Jupyter was launched from
    (e.g. launched from repo root, from backend/, or from backend/notebooks/)."""
    candidates = [start] + list(start.parents)
    for p in candidates:
        maybe = p if p.name == "backend" else p / "backend"
        if (maybe / "ingestion").is_dir() and (maybe / ".env.example").exists():
            return maybe
    raise RuntimeError(
        f"Could not locate 'backend/' above {start}. "
        "Run this notebook from within the f1-strategy-lab repo."
    )

NOTEBOOK = Path.cwd()
BACKEND = find_backend_dir(NOTEBOOK)
PROJECT_ROOT = BACKEND.parent

# Allow imports like: from ml.tire_degradation.features import ...
sys.path.insert(0, str(BACKEND))

from dotenv import load_dotenv

env_path = BACKEND / ".env"
load_dotenv(env_path)

import pandas as pd
from sqlalchemy import create_engine, text
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

from ml.tire_degradation.features import build_training_frame

DATABASE_URL = os.getenv("DATABASE_URL")

if DATABASE_URL is None:
    raise RuntimeError(
        f"DATABASE_URL not found. Looked for it in {env_path}. "
        f"Does that file exist and does it define DATABASE_URL=...?"
    )

engine = create_engine(DATABASE_URL)
print(f"BACKEND resolved to: {BACKEND}")
print(f"Loaded .env from:    {env_path}  (exists: {env_path.exists()})")
print("Engine created OK.")


BACKEND resolved to: c:\Users\ASUS\Desktop\6-months projects\week 6 - end of phase 1\f1-strategy-lab\backend
Loaded .env from:    c:\Users\ASUS\Desktop\6-months projects\week 6 - end of phase 1\f1-strategy-lab\backend\.env  (exists: True)
Engine created OK.


## Step 1 — Pull all lap data for the 2023 season
Join laps → races → drivers. Filter out laps that can't be used as clean degradation
signal: null lap times, null/unknown compound, and non-green track status (SC/VSC/red
compress lap times and would poison the degradation signal).


**Stopgap included below:** `compound != 'None'` is filtered explicitly, since `ingest.py` currently writes the literal string `"None"` for some laps instead of a true SQL NULL (see `CHECKPOINT_DATA_EXPLORATION_CLEANING.md`). This can be removed once that bug is fixed at the source and affected races are re-ingested.

In [12]:
query = text("""
    SELECT
        l.race_id,
        r.round,
        r.circuit,
        r.race_date,
        l.driver_id,
        l.lap_number,
        l.lap_time_ms,
        l.compound,
        l.tire_age,
        l.stint_number,
        l.track_status,
        l.ambient_temp_c,
        l.track_temp_c,
        l.rainfall
    FROM laps l
    JOIN races r ON r.id = l.race_id
    WHERE l.lap_time_ms IS NOT NULL
      AND l.compound IS NOT NULL AND l.compound != 'UNKNOWN' AND l.compound != 'None'
      AND l.track_status = '1'
    ORDER BY r.round, l.driver_id, l.lap_number
""")
raw = pd.read_sql(query, engine)
print(f"Rows: {len(raw)}")
print(f"Races represented: {raw['round'].nunique()} / 22")
raw.head()

Rows: 21890
Races represented: 22 / 22


,race_id,round,circuit,race_date,driver_id,lap_number,lap_time_ms,compound,tire_age,stint_number,track_status,ambient_temp_c,track_temp_c,rainfall
0,1,1,Sakhir,2023-03-05,1,3,98006,SOFT,6,1,1,27.3,31.2,False
1,1,1,Sakhir,2023-03-05,1,4,97976,SOFT,7,1,1,27.3,31.2,False
2,1,1,Sakhir,2023-03-05,1,5,98035,SOFT,8,1,1,27.2,31.0,False
3,1,1,Sakhir,2023-03-05,1,6,97986,SOFT,9,1,1,27.1,31.0,False
4,1,1,Sakhir,2023-03-05,1,7,98021,SOFT,10,1,1,27.1,30.9,False


## Step 2 — Engineer the target
Target = `lap_time_delta` = this lap's time minus the **fresh-tire baseline** for
that compound at that circuit. Fresh-tire baseline = median lap time on `tire_age <= 2`
for that (circuit, compound) pair, computed across the full season.

Note on the full-season-baseline choice: this is a deliberate simplification for the
notebook exercise. `features.py`'s real `compute_baselines()` should document the
tradeoff explicitly (a baseline computed only from the train split would be more
rigorous but noisier, since fresh-tire laps are a small fraction of total laps per
circuit). For now, keep the same simplification here so this notebook's numbers are
comparable to what `train.py` will produce.

In [13]:
def compute_baselines(df: pd.DataFrame) -> pd.DataFrame:
    fresh = df[df["tire_age"] <= 2]
    baselines = (
        fresh.groupby(["circuit", "compound"])["lap_time_ms"]
        .median()
        .reset_index()
        .rename(columns={"lap_time_ms": "baseline_ms"})
    )
    return baselines

baselines = compute_baselines(raw)
print(f"Baselines computed for {len(baselines)} (circuit, compound) pairs")
baselines.head()

Baselines computed for 63 (circuit, compound) pairs


,circuit,compound,baseline_ms
0,Austin,HARD,109257.0
1,Austin,MEDIUM,105393.0
2,Austin,SOFT,122300.0
3,Baku,HARD,113128.0
4,Baku,MEDIUM,108818.0


In [14]:
data = raw.merge(baselines, on=["circuit", "compound"], how="inner")
data["lap_time_delta"] = data["lap_time_ms"] - data["baseline_ms"]

# Rows with no baseline (circuit/compound combo never seen fresh, e.g. a compound
# only used briefly) get dropped by the inner merge above — check how many:
print(f"Rows before baseline merge: {len(raw)}")
print(f"Rows after baseline merge:  {len(data)}  ({len(raw) - len(data)} dropped, no baseline)")

data[["circuit", "compound", "tire_age", "lap_time_ms", "baseline_ms", "lap_time_delta"]].head()

Rows before baseline merge: 21890
Rows after baseline merge:  21804  (86 dropped, no baseline)


,circuit,compound,tire_age,lap_time_ms,baseline_ms,lap_time_delta
0,Sakhir,SOFT,6,98006,98503.0,-497.0
1,Sakhir,SOFT,7,97976,98503.0,-527.0
2,Sakhir,SOFT,8,98035,98503.0,-468.0
3,Sakhir,SOFT,9,97986,98503.0,-517.0
4,Sakhir,SOFT,10,98021,98503.0,-482.0


## Step 3 — Feature matrix

In [15]:
circuit_map = {c: i for i, c in enumerate(sorted(data["circuit"].unique()))}
data["circuit_code"] = data["circuit"].map(circuit_map)

compound_dummies = pd.get_dummies(data["compound"], prefix="compound")
feature_cols = ["tire_age", "lap_number", "circuit_code", "ambient_temp_c", "track_temp_c"]
X = pd.concat([data[feature_cols], compound_dummies], axis=1)
y = data["lap_time_delta"]

X = X.fillna(X.median(numeric_only=True))
print(f"Feature matrix: {X.shape}")
X.head()

Feature matrix: (21804, 10)


,tire_age,lap_number,circuit_code,ambient_temp_c,track_temp_c,compound_HARD,compound_INTERMEDIATE,compound_MEDIUM,compound_SOFT,compound_WET
0,6,3,14,27.3,31.2,False,False,False,True,False
1,7,4,14,27.3,31.2,False,False,False,True,False
2,8,5,14,27.2,31.0,False,False,False,True,False
3,9,6,14,27.1,31.0,False,False,False,True,False
4,10,7,14,27.1,30.9,False,False,False,True,False


## Step 4 — Deliberately wrong: naive random 80/20 split
This is the mistake to make on purpose, so the next section can show what it looks
like when it goes right.

In [16]:
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_naive = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
model_naive.fit(X_train_naive, y_train_naive)

pred_naive = model_naive.predict(X_test_naive)
mae_naive = mean_absolute_error(y_test_naive, pred_naive)
print(f"Naive (random split) MAE: {mae_naive:.2f} ms")

Naive (random split) MAE: 1064.42 ms


### Why this MAE looks suspiciously good — the leakage, in plain terms

A random row-level split doesn't respect that laps are *grouped* by race. With
`shuffle=True` (the default), laps from the same race — often from the same driver's
stint — land on both sides of the split. The model isn't being asked "can you predict
degradation for a race you've never seen"; it's being asked "here's 80% of a race's
laps, predict the other 20% from the *same* stint, same track conditions, same driver
pattern, that day." That's closer to interpolation within an almost-fully-observed
curve than genuine generalization to a new circuit or a new race weekend.

The gap between this number and the honest one below **is** the size of the leak —
not a rounding difference, a real measure of how much the model was cheating.

## Step 5 — Fix it: time-aware split
Sort by race round, train on rounds 1–15, validate on rounds 16–22. Never shuffle by
lap — that leaks future-race circuit knowledge (and even future car-development-phase
pace trends) into training.

In [17]:
train_rounds = list(range(1, 16))
val_rounds = list(range(16, 23))

train_mask = data["round"].isin(train_rounds)
val_mask = data["round"].isin(val_rounds)

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]

print(f"Train rows: {len(X_train)} (rounds {train_rounds[0]}-{train_rounds[-1]})")
print(f"Val rows:   {len(X_val)} (rounds {val_rounds[0]}-{val_rounds[-1]})")

model = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

pred = model.predict(X_val)
mae_honest = mean_absolute_error(y_val, pred)
print(f"Honest (time-aware split) MAE: {mae_honest:.2f} ms")
print(f"Naive MAE was:                 {mae_naive:.2f} ms")
print(f"Leakage gap:                   {mae_honest - mae_naive:.2f} ms")

Train rows: 15167 (rounds 1-15)
Val rows:   6637 (rounds 16-22)
Honest (time-aware split) MAE: 7237.77 ms
Naive MAE was:                 1064.42 ms
Leakage gap:                   6173.35 ms


**This honest number — not the naive one — is what goes into `ML_FINDINGS.md`.**

## Step 6 — Breakdown by compound

In [18]:
val_results = data[val_mask].copy()
val_results["pred"] = pred
val_results["abs_error"] = (val_results["lap_time_delta"] - val_results["pred"]).abs()

val_results.groupby("compound")["abs_error"].agg(["mean", "median", "count"]).rename(columns={"mean": "MAE"})

,MAE,median,count
compound,,,
HARD,7780.653322,7981.188114,3107
MEDIUM,7560.963416,5943.329617,2741
SOFT,3977.201917,2794.052097,789


## Step 7 — Breakdown by circuit\nNote which circuits the model struggles with — usually low-lap-count or unusual-degradation circuits (street circuits, resurfaced tracks).

In [19]:
circuit_mae = val_results.groupby("circuit")["abs_error"].agg(["mean", "count"]).rename(columns={"mean": "MAE"})
circuit_mae.sort_values("MAE", ascending=False)

,MAE,count
circuit,,
Suzuka,10721.309140,749
Las Vegas,9750.920221,670
Yas Island,8671.670105,1157
Lusail,7809.261369,896
Mexico City,6759.307203,1101
São Paulo,5726.700168,1057
Austin,2927.890772,1007


## Limitations
Carry these into `ML_FINDINGS.md` alongside the honest MAE:

- **Fuel-load confound.** `lap_number` is used as a fuel burn-off proxy, but it's
  confounded with genuine tire degradation — both correlate with lap number, and
  there's no fuel-mass data available to separate the two effects.
- **`circuit_code` is an arbitrary integer.** A tree model can't infer that "Monza is
  a low-downforce power circuit like Baku" from an integer label. Generalization to a
  circuit not seen in training is genuinely limited by this encoding.
- **WET/INTERMEDIATE are underrepresented.** One season has very few wet-race laps —
  treat the per-compound MAE for those two with low confidence regardless of what the
  number says.
- **Safety car / VSC laps excluded, not modeled.** This notebook filters to
  `track_status = '1'` only, so the model has no signal at all for degradation
  behavior during or immediately after SC/VSC periods.
- **Rookie drivers have thin data.** Any driver who only raced part of the season
  contributes far fewer laps than a full-season driver, which the model doesn't
  account for.
- **Baselines computed across the full season**, not just the training split — see
  the note in Step 2. `features.py`'s docstring should carry this rationale forward.


## Step 8 — Serialize the model bundle\nThe `circuit_map` travels with the model — recomputing it on a different season/race set would silently reassign integers and break inference.

In [20]:
import joblib
from pathlib import Path

model_dir = BACKEND / "ml" / "tire_degradation"
model_dir.mkdir(parents=True, exist_ok=True)

bundle = {"model": model, "circuit_map": circuit_map, "feature_cols": list(X.columns)}
joblib.dump(bundle, model_dir / "model.pkl")
print(f"Saved bundle to {model_dir / 'model.pkl'}")

Saved bundle to c:\Users\ASUS\Desktop\6-months projects\week 6 - end of phase 1\f1-strategy-lab\backend\ml\tire_degradation\model.pkl


## Next: extract into `features.py` / `train.py`
Per the project discipline — this notebook proved the idea and worked out the leakage
story by hand. The reusable pieces (`compute_baselines`, the feature-building logic in
Step 3, the time-aware split boundary) now belong in `ml/tire_degradation/features.py`,
not duplicated here. `train.py` should import from there and reproduce Steps 1–8
end-to-end, appending its result to `ML_FINDINGS.md`. Then `model.py` wraps the saved
bundle behind `TireDegradationModel.predict(features) -> float`.
